# 🚀 Vectyfi Radar — ML Model for EU Public Tender Prediction

**Goal:** Predict whether a public tender will result in a contract award.

**Dataset:** TED (Tenders Electronic Daily) — Contract Forming Cycle 2018–2023

**ML Task:** Binary Classification → awarded (1) vs. not awarded (0)

# Some notes about the data
- thresholds? filter all data for these thresholds? https://single-market-economy.ec.europa.eu/single-market/public-procurement/legal-rules-and-implementation/thresholds_en
- EU Tenders Electronic Daily (TED) website: https://data.europa.eu/data/datasets/ted-csv?locale=en
    - stored the data docs/overviews in folder (in .gitignore): radar.vectyfi/docs
    - Overview of the data set in PDF (on the website scroll down to 'Documentation'), file "TED(csv)_data_information_v3.6.1 pdf version": https://data.europa.eu/api/hub/store/data/6937ff8799566d7a3a5f93ff
    - Overview of the data quality as PPT file "20140429_ESWG_Data_Quality": https://data.europa.eu/euodp/repository/ec/dg-grow/mapps/20140429_ESWG_Data_Quality.ppt
    - Description of missing values and outliers file "20140429_ESWG_Varela-Irimia": https://data.europa.eu/euodp/repository/ec/dg-grow/mapps/20140429_ESWG_Varela-Irimia.pdf
    - Advanced notes on methodology file "Advanced notes on methodology - Version 0.92": http://data.europa.eu/euodp/en/data/storage/f/2022-02-14T122830/TED_advanced_notes_vers_0.92.pdf
- there are two different file types, downloaded both of them:
    - Contract notices: before the award of the tenders
        - file "TED - Contract notices 2018-2023": https://data.europa.eu/api/hub/store/data/ted-contract-notices-2018-2023.zip
        - unpacked: *export_CFC_2018_2023.csv*
        - should not have data leakage
    - Contract award notices: after the award of the tenders
        - file "TED - Contract award notices 2018-2023": https://data.europa.eu/api/hub/store/data/ted-contract-award-notices-2018-2023.zip
        - unpacked: *export_CAN_2023_2018.csv*
        - **however**, columns have data leakage as filled after awarding the tenders -> filter accordingly!
        - **USE** this file, should have the feature for prediction and 
- the data set on Kaggle (with a simple Jupyter notebook to check which country had how many tenders): https://www.kaggle.com/datasets/nartaa/tenders-electronic-daily-2018-2023

 ---
 ## 1. Setup & Imports

 First, we load all the libraries we need. If any are missing: `pip install <name>`

In [1]:
# %%
# Core
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, accuracy_score, f1_score, recall_score,
    precision_score, precision_recall_curve, fbeta_score
)

# Settings
pd.set_option('display.max_columns', 70)
pd.set_option('display.max_columns', 100)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('viridis')

print('✅ All imports loaded successfully!')

# XGBoost
try:
    from xgboost import XGBClassifier
    print('✅ XGBoost loaded')
except ImportError:
    print('⚠️ XGBoost not installed. Run: pip install xgboost')

✅ All imports loaded successfully!
✅ XGBoost loaded


# Raw data loading and check

In [ ]:
# Load all data to create smaller sample data sets
df_raw_all = pd.read_csv('../raw_data/export_CAN_2023_2018.csv')

In [33]:
df_raw_all.head(3)

,ID_NOTICE_CAN,TED_NOTICE_URL,YEAR,ID_TYPE,DT_DISPATCH,XSD_VERSION,CANCELLED,CORRECTIONS,B_MULTIPLE_CAE,CAE_NAME,CAE_NATIONALID,CAE_ADDRESS,CAE_TOWN,CAE_POSTAL_CODE,CAE_GPA_ANNEX,ISO_COUNTRY_CODE,ISO_COUNTRY_CODE_GPA,B_MULTIPLE_COUNTRY,ISO_COUNTRY_CODE_ALL,CAE_TYPE,EU_INST_CODE,MAIN_ACTIVITY,B_ON_BEHALF,B_INVOLVES_JOINT_PROCUREMENT,B_AWARDED_BY_CENTRAL_BODY,TYPE_OF_CONTRACT,TAL_LOCATION_NUTS,B_FRA_AGREEMENT,FRA_ESTIMATED,B_FRA_CONTRACT,B_DYN_PURCH_SYST,CPV,MAIN_CPV_CODE_GPA,ID_LOT,ADDITIONAL_CPVS,B_GPA,GPA_COVERAGE,LOTS_NUMBER,VALUE_EURO,VALUE_EURO_FIN_1,VALUE_EURO_FIN_2,B_EU_FUNDS,TOP_TYPE,B_ACCELERATED,OUT_OF_DIRECTIVES,CRIT_CODE,CRIT_PRICE_WEIGHT,CRIT_CRITERIA,CRIT_WEIGHTS,B_ELECTRONIC_AUCTION,NUMBER_AWARDS,ID_AWARD,ID_LOT_AWARDED,INFO_ON_NON_AWARD,INFO_UNPUBLISHED,B_AWARDED_TO_A_GROUP,WIN_NAME,WIN_NATIONALID,WIN_ADDRESS,WIN_TOWN,WIN_POSTAL_CODE,WIN_COUNTRY_CODE,B_CONTRACTOR_SME,CONTRACT_NUMBER,TITLE,NUMBER_OFFERS,NUMBER_TENDERS_SME,NUMBER_TENDERS_OTHER_EU,NUMBER_TENDERS_NON_EU,NUMBER_OFFERS_ELECTR,AWARD_EST_VALUE_EURO,AWARD_VALUE_EURO,AWARD_VALUE_EURO_FIN_1,B_SUBCONTRACTED,DT_AWARD
0,20184,ted.europa.eu/udl?uri=TED:NOTICE:4-2018:TEXT:E...,2018,3,22/12/17,R209.S2,0,0,N,European Insurance and Occupational Pensions A...,NaN,"WesthafenTower, Westhafenplatz 1",Frankfurt am Main,60327,A1,DE,EU,N,NaN,5,AG,General public\services,N,N,N,S,DE712,Y,NaN,Y,N,72300000,72300.0,2,72319000,Y,6.0,1.0,NaN,NaN,NaN,Y,OPE,NaN,0,M,30,Technical quality,70,N,1,8447164.0,2,PROCUREMENT_UNSUCCESSFUL,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"EIOPA/OP/009/2017, Lot 2",Financial Market Input Data for Validation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN
1,20185,ted.europa.eu/udl?uri=TED:NOTICE:5-2018:TEXT:E...,2018,3,22/12/17,R209.S2,0,0,N,European Food Safety Authority (EFSA),NaN,Via Carlo Magno 1A,Parma,43126,A1,IT,EU,N,NaN,5,AG,General public\services,N,N,N,S,ITH52,Y,NaN,Y,N,79411000,79411.0,NaN,NaN,N,2.0,1.0,1500000.0,1500000.0,1500000.0,N,OPE,NaN,0,M,NaN,Cheapest price offer/price of tender X---Measu...,40---60,N,2,8447165.0,NaN,NaN,N,Y,ELT associati di Guaglio Davide e Molinari Pao...,NaN,Via Mantovanella 4---Galleria Crocetta 10/a---...,Mantova---Parma---Parma---Parma---Parma,46100---43126---43126---43123---43122,IT---IT---IT---IT---IT,Y---Y---Y---Y---Y,OC/EFSA/CORSER/2017/03 - FWC 01,Consultancy services to improve efficiency and...,2.0,2.0,0.0,0.0,NaN,1500000.0,1500000.0,1500000.0,N,18/12/17
2,20185,ted.europa.eu/udl?uri=TED:NOTICE:5-2018:TEXT:E...,2018,3,22/12/17,R209.S2,0,0,N,European Food Safety Authority (EFSA),NaN,Via Carlo Magno 1A,Parma,43126,A1,IT,EU,N,NaN,5,AG,General public\services,N,N,N,S,ITH52,Y,NaN,Y,N,79411000,79411.0,NaN,NaN,N,2.0,1.0,1500000.0,1500000.0,1500000.0,N,OPE,NaN,0,M,NaN,Cheapest price offer/price of tender X---Measu...,40---60,N,2,8447166.0,NaN,NaN,N,N,A.I.Erre engineering,NaN,Strada Cavagnari 10,Parma,43126,IT,Y,OC/EFSA/CORSER/2017/03 - FWC02,Consultancy service to improve efficiency and ...,2.0,2.0,0.0,0.0,NaN,1500000.0,1500000.0,1500000.0,N,18/12/17


In [34]:
df_raw_all.tail(3)

,ID_NOTICE_CAN,TED_NOTICE_URL,YEAR,ID_TYPE,DT_DISPATCH,XSD_VERSION,CANCELLED,CORRECTIONS,B_MULTIPLE_CAE,CAE_NAME,CAE_NATIONALID,CAE_ADDRESS,CAE_TOWN,CAE_POSTAL_CODE,CAE_GPA_ANNEX,ISO_COUNTRY_CODE,ISO_COUNTRY_CODE_GPA,B_MULTIPLE_COUNTRY,ISO_COUNTRY_CODE_ALL,CAE_TYPE,EU_INST_CODE,MAIN_ACTIVITY,B_ON_BEHALF,B_INVOLVES_JOINT_PROCUREMENT,B_AWARDED_BY_CENTRAL_BODY,TYPE_OF_CONTRACT,TAL_LOCATION_NUTS,B_FRA_AGREEMENT,FRA_ESTIMATED,B_FRA_CONTRACT,B_DYN_PURCH_SYST,CPV,MAIN_CPV_CODE_GPA,ID_LOT,ADDITIONAL_CPVS,B_GPA,GPA_COVERAGE,LOTS_NUMBER,VALUE_EURO,VALUE_EURO_FIN_1,VALUE_EURO_FIN_2,B_EU_FUNDS,TOP_TYPE,B_ACCELERATED,OUT_OF_DIRECTIVES,CRIT_CODE,CRIT_PRICE_WEIGHT,CRIT_CRITERIA,CRIT_WEIGHTS,B_ELECTRONIC_AUCTION,NUMBER_AWARDS,ID_AWARD,ID_LOT_AWARDED,INFO_ON_NON_AWARD,INFO_UNPUBLISHED,B_AWARDED_TO_A_GROUP,WIN_NAME,WIN_NATIONALID,WIN_ADDRESS,WIN_TOWN,WIN_POSTAL_CODE,WIN_COUNTRY_CODE,B_CONTRACTOR_SME,CONTRACT_NUMBER,TITLE,NUMBER_OFFERS,NUMBER_TENDERS_SME,NUMBER_TENDERS_OTHER_EU,NUMBER_TENDERS_NON_EU,NUMBER_OFFERS_ELECTR,AWARD_EST_VALUE_EURO,AWARD_VALUE_EURO,AWARD_VALUE_EURO_FIN_1,B_SUBCONTRACTED,DT_AWARD
6198060,2023794676,ted.europa.eu/udl?uri=TED:NOTICE:794676-2023:T...,2023,18,27/12/23,R208.S5,0,0,NaN,Országos Idegenrendészeti Főigazgatóság,EKRSZ_68188616,Budafoki út 60.,Budapest,1117,NaN,HU,NaN,NaN,NaN,N,NaN,Public Order and Safety,N,NaN,NaN,S,NaN,N,NaN,N,NaN,79821000,NaN,NaN,NaN,NaN,NaN,3.0,4130249.53,4130249.53,4130249.53,N,NIP,NaN,0,L,100,NaN,NaN,N,3,17001230.0,1,NaN,N,NaN,ANY Biztonsági Nyomda Nyrt.,NaN,Halom utca 5.,Budapest,1102,HU,NaN,106-G-50984/2023,Egységes EU formátumú tartózkodási engedély ok...,1.0,NaN,NaN,NaN,0.0,NaN,2578701.22,2578701.22,N,19/12/23
6198061,2023794676,ted.europa.eu/udl?uri=TED:NOTICE:794676-2023:T...,2023,18,27/12/23,R208.S5,0,0,NaN,Országos Idegenrendészeti Főigazgatóság,EKRSZ_68188616,Budafoki út 60.,Budapest,1117,NaN,HU,NaN,NaN,NaN,N,NaN,Public Order and Safety,N,NaN,NaN,S,NaN,N,NaN,N,NaN,79821000,NaN,NaN,NaN,NaN,NaN,3.0,4130249.53,4130249.53,4130249.53,N,NIP,NaN,0,L,100,NaN,NaN,N,3,17001231.0,2,NaN,N,NaN,ANY Biztonsági Nyomda Nyrt.,NaN,Halom utca 5.,Budapest,1102,HU,NaN,106-G-50984/1/2023,Vízumbélyeg okmányok előállítása,2.0,NaN,NaN,NaN,0.0,NaN,976666.92,976666.92,Y,19/12/23
6198062,2023794676,ted.europa.eu/udl?uri=TED:NOTICE:794676-2023:T...,2023,18,27/12/23,R208.S5,0,0,NaN,Országos Idegenrendészeti Főigazgatóság,EKRSZ_68188616,Budafoki út 60.,Budapest,1117,NaN,HU,NaN,NaN,NaN,N,NaN,Public Order and Safety,N,NaN,NaN,S,NaN,N,NaN,N,NaN,79821000,NaN,NaN,NaN,NaN,NaN,3.0,4130249.53,4130249.53,4130249.53,N,NIP,NaN,0,L,100,NaN,NaN,N,3,17001232.0,3,NaN,N,NaN,ANY Biztonsági Nyomda Nyrt.,NaN,Halom utca 5.,Budapest,1102,HU,NaN,106-G-50984/2/2023,„A” és „B” okmányvédelmi kategóriába sorolt ok...,1.0,NaN,NaN,NaN,0.0,NaN,574881.39,574881.39,N,19/12/23


In [35]:
# not sure why null values not shown (but I think here dtypes have to be defined)
df_raw_all.info() # memory usage: 3.7+ GB

<class 'pandas.DataFrame'>
RangeIndex: 6198063 entries, 0 to 6198062
Data columns (total 75 columns):
 #   Column                        Dtype  
---  ------                        -----  
 0   ID_NOTICE_CAN                 int64  
 1   TED_NOTICE_URL                str    
 2   YEAR                          int64  
 3   ID_TYPE                       int64  
 4   DT_DISPATCH                   str    
 5   XSD_VERSION                   str    
 6   CANCELLED                     int64  
 7   CORRECTIONS                   int64  
 8   B_MULTIPLE_CAE                str    
 9   CAE_NAME                      str    
 10  CAE_NATIONALID                str    
 11  CAE_ADDRESS                   str    
 12  CAE_TOWN                      str    
 13  CAE_POSTAL_CODE               str    
 14  CAE_GPA_ANNEX                 str    
 15  ISO_COUNTRY_CODE              str    
 16  ISO_COUNTRY_CODE_GPA          str    
 17  B_MULTIPLE_COUNTRY            str    
 18  ISO_COUNTRY_CODE_ALL          str

In [36]:
df_raw_all.shape

(6198063, 75)

In [37]:
pd.options.display.min_rows = 200
df_raw_all.isnull().sum()

ID_NOTICE_CAN                         0
TED_NOTICE_URL                        0
YEAR                                  0
ID_TYPE                               0
DT_DISPATCH                           0
XSD_VERSION                           0
CANCELLED                             0
CORRECTIONS                           0
B_MULTIPLE_CAE                   108090
CAE_NAME                              0
CAE_NATIONALID                  2204617
CAE_ADDRESS                      102302
CAE_TOWN                              2
CAE_POSTAL_CODE                  116314
CAE_GPA_ANNEX                   1101251
ISO_COUNTRY_CODE                      0
ISO_COUNTRY_CODE_GPA            1101251
B_MULTIPLE_COUNTRY               108090
ISO_COUNTRY_CODE_ALL            6195379
CAE_TYPE                              0
EU_INST_CODE                    6183636
MAIN_ACTIVITY                        54
B_ON_BEHALF                       11929
B_INVOLVES_JOINT_PROCUREMENT     119533
B_AWARDED_BY_CENTRAL_BODY        119533


# Load data again, this time define the data types

In [3]:
# define the data column data types
CAN_DTYPES: dict = {
    # ── Notice metadata ────────────────────────────────────────────────────────
    "ID_NOTICE_CAN":                "int32",
    "TED_NOTICE_URL":               "str",
    "YEAR":                         "int16",
    "ID_TYPE":                      "int8",
    "DT_DISPATCH":                  "str",      # parsed below → datetime
    "XSD_VERSION":                  "category",
    "CANCELLED":                    "int8",     # 0 / 1
    "CORRECTIONS":                  "int8",

    # ── Contracting authority ──────────────────────────────────────────────────
    "B_MULTIPLE_CAE":               "category",
    "CAE_NAME":                     "str",
    "CAE_NATIONALID":               "str",      # mixed formats (VAT, reg. no.)
    "CAE_ADDRESS":                  "str",
    "CAE_TOWN":                     "str",
    "CAE_POSTAL_CODE":              "str",      # leading zeros → keep as str
    "CAE_GPA_ANNEX":                "category",
    "ISO_COUNTRY_CODE":             "category",
    "ISO_COUNTRY_CODE_GPA":         "category",
    "B_MULTIPLE_COUNTRY":           "category",
    "ISO_COUNTRY_CODE_ALL":         "str",      # ~100 % null; NUTS multi-country
    "CAE_TYPE":                     "category", # 1,3,4,5,5A,6,8,N,R,Z
    "EU_INST_CODE":                 "category", # ~99 % null; AG,BC,EC,EP,...
    "MAIN_ACTIVITY":                "category",
    "B_ON_BEHALF":                  "category",
    "B_INVOLVES_JOINT_PROCUREMENT": "category",
    "B_AWARDED_BY_CENTRAL_BODY":    "category",

    # ── Contract / procedure characteristics ───────────────────────────────────
    "TYPE_OF_CONTRACT":             "category", # W / U / S
    "TAL_LOCATION_NUTS":            "str",      # can hold comma-separated codes
    "B_FRA_AGREEMENT":              "category",
    "FRA_ESTIMATED":                "category", # K / A / KA / C
    "B_FRA_CONTRACT":               "category",
    "B_DYN_PURCH_SYST":             "category",
    "CPV":                          "int32",
    "MAIN_CPV_CODE_GPA":            "float32",  # ~6.5 % null
    "ID_LOT":                       "str",      # '1','2',… but not always numeric
    "ADDITIONAL_CPVS":              "str",      # '---'-separated list of CPV codes
    "B_GPA":                        "category",
    "GPA_COVERAGE":                 "float32",  # codes 1–6, ~6.5 % null
    "LOTS_NUMBER":                  "float32",  # nullable count
    "VALUE_EURO":                   "float64",
    "VALUE_EURO_FIN_1":             "float64",
    "VALUE_EURO_FIN_2":             "float64",
    "B_EU_FUNDS":                   "category",
    "TOP_TYPE":                     "category", # OPE/RES/COD/NIC/NOC/AWP/INP
    "B_ACCELERATED":                "category", # ~96 % null; Y when used
    "OUT_OF_DIRECTIVES":            "int8",     # 0 / 1
    "CRIT_CODE":                    "category", # L / M
    "CRIT_PRICE_WEIGHT":            "str",      # dirty: '30','30 %','60,00','50/50'
    "CRIT_CRITERIA":                "str",      # '---'-separated criterion names
    "CRIT_WEIGHTS":                 "str",      # '---'-separated weights
    "B_ELECTRONIC_AUCTION":         "category",
    "NUMBER_AWARDS":                "int16",

    # ── Award metadata ─────────────────────────────────────────────────────────
    "ID_AWARD":                     "Int64",    # nullable; ~13 % null (non-awards)
    "ID_LOT_AWARDED":               "str",
    "INFO_ON_NON_AWARD":            "str", # changed from 'category' to 'str' so don't have to do it below; null=awarded / UNSUCCESSFUL / DISCONTINUED
    "INFO_UNPUBLISHED":             "category",
    "B_AWARDED_TO_A_GROUP":         "category",

    # ── Winner identity ────────────────────────────────────────────────────────
    # ⚠ Consortium rows (B_AWARDED_TO_A_GROUP=Y) store values as 'A---B---C'.
    # Parse with _parse_consortium_sme() below before using as features/target.
    "WIN_NAME":                     "str",
    "WIN_NATIONALID":               "str",
    "WIN_ADDRESS":                  "str",
    "WIN_TOWN":                     "str",
    "WIN_POSTAL_CODE":              "str",
    "WIN_COUNTRY_CODE":             "str",
    "B_CONTRACTOR_SME":             "str",      # '---'-concatenated for consortia

    # ── Other CA-level (post-award — exclude from model features) ──────────────
    "CONTRACT_NUMBER":              "str",
    "TITLE":                        "str",
    "NUMBER_OFFERS":                "float32",  # nullable count
    "NUMBER_TENDERS_SME":           "float32",
    "NUMBER_TENDERS_OTHER_EU":      "float32",
    "NUMBER_TENDERS_NON_EU":        "float32",
    "NUMBER_OFFERS_ELECTR":         "float32",
    "AWARD_EST_VALUE_EURO":         "float64",
    "AWARD_VALUE_EURO":             "float64",
    "AWARD_VALUE_EURO_FIN_1":       "float64",
    "B_SUBCONTRACTED":              "category",
    "DT_AWARD":                     "str",      # parsed below → datetime
}

In [4]:
# Date columns and their format (DD/MM/YY, two-digit year)
_DATE_COLS = ["DT_DISPATCH", "DT_AWARD"]
_DATE_FORMAT = "%d/%m/%y"

In [5]:
def load_raw_can(path: str | Path | None = None, nrows: int | None = None) -> pd.DataFrame:
    """Load the TED Contract Award Notices CSV with correct dtypes.

    Args:
        path:   Path to the CSV. Defaults to raw_data/export_CAN_2023_2018.csv.
        nrows:  Limit rows for rapid iteration (e.g. nrows=10_000 for EDA).
                Set to None to load the full ~4 GB file.

    Returns:
        DataFrame with explicit dtypes and parsed datetime columns.
        Row = one contract award (ID_AWARD level).
    """
    if path is None:
        path = Path(__file__).resolve().parents[1] / "raw_data" / "export_CAN_2023_2018.csv"

    df = pd.read_csv(
        path,
        dtype=CAN_DTYPES,
        nrows=nrows,
        encoding="utf-8",
        low_memory=False,   # redundant once dtype is explicit, but safe to keep
    )

    for col in _DATE_COLS:
        df[col] = pd.to_datetime(df[col], format=_DATE_FORMAT, dayfirst=True, errors="coerce")

    return df

In [6]:
df_raw_all = load_raw_can(path='../raw_data/export_CAN_2023_2018.csv', nrows=None)

In [7]:
df_raw_all.info()

<class 'pandas.DataFrame'>
RangeIndex: 6198063 entries, 0 to 6198062
Data columns (total 75 columns):
 #   Column                        Dtype         
---  ------                        -----         
 0   ID_NOTICE_CAN                 int32         
 1   TED_NOTICE_URL                str           
 2   YEAR                          int16         
 3   ID_TYPE                       int8          
 4   DT_DISPATCH                   datetime64[us]
 5   XSD_VERSION                   category      
 6   CANCELLED                     int8          
 7   CORRECTIONS                   int8          
 8   B_MULTIPLE_CAE                category      
 9   CAE_NAME                      str           
 10  CAE_NATIONALID                str           
 11  CAE_ADDRESS                   str           
 12  CAE_TOWN                      str           
 13  CAE_POSTAL_CODE               str           
 14  CAE_GPA_ANNEX                 category      
 15  ISO_COUNTRY_CODE              category     

In [8]:
df_raw_all.shape

(6198063, 75)

In [9]:
df_raw_all.head(3)

,ID_NOTICE_CAN,TED_NOTICE_URL,YEAR,ID_TYPE,DT_DISPATCH,XSD_VERSION,CANCELLED,CORRECTIONS,B_MULTIPLE_CAE,CAE_NAME,CAE_NATIONALID,CAE_ADDRESS,CAE_TOWN,CAE_POSTAL_CODE,CAE_GPA_ANNEX,ISO_COUNTRY_CODE,ISO_COUNTRY_CODE_GPA,B_MULTIPLE_COUNTRY,ISO_COUNTRY_CODE_ALL,CAE_TYPE,EU_INST_CODE,MAIN_ACTIVITY,B_ON_BEHALF,B_INVOLVES_JOINT_PROCUREMENT,B_AWARDED_BY_CENTRAL_BODY,TYPE_OF_CONTRACT,TAL_LOCATION_NUTS,B_FRA_AGREEMENT,FRA_ESTIMATED,B_FRA_CONTRACT,B_DYN_PURCH_SYST,CPV,MAIN_CPV_CODE_GPA,ID_LOT,ADDITIONAL_CPVS,B_GPA,GPA_COVERAGE,LOTS_NUMBER,VALUE_EURO,VALUE_EURO_FIN_1,VALUE_EURO_FIN_2,B_EU_FUNDS,TOP_TYPE,B_ACCELERATED,OUT_OF_DIRECTIVES,CRIT_CODE,CRIT_PRICE_WEIGHT,CRIT_CRITERIA,CRIT_WEIGHTS,B_ELECTRONIC_AUCTION,NUMBER_AWARDS,ID_AWARD,ID_LOT_AWARDED,INFO_ON_NON_AWARD,INFO_UNPUBLISHED,B_AWARDED_TO_A_GROUP,WIN_NAME,WIN_NATIONALID,WIN_ADDRESS,WIN_TOWN,WIN_POSTAL_CODE,WIN_COUNTRY_CODE,B_CONTRACTOR_SME,CONTRACT_NUMBER,TITLE,NUMBER_OFFERS,NUMBER_TENDERS_SME,NUMBER_TENDERS_OTHER_EU,NUMBER_TENDERS_NON_EU,NUMBER_OFFERS_ELECTR,AWARD_EST_VALUE_EURO,AWARD_VALUE_EURO,AWARD_VALUE_EURO_FIN_1,B_SUBCONTRACTED,DT_AWARD
0,20184,ted.europa.eu/udl?uri=TED:NOTICE:4-2018:TEXT:E...,2018,3,2017-12-22,R209.S2,0,0,N,European Insurance and Occupational Pensions A...,NaN,"WesthafenTower, Westhafenplatz 1",Frankfurt am Main,60327,A1,DE,EU,N,NaN,5,AG,General public\services,N,N,N,S,DE712,Y,NaN,Y,N,72300000,72300.0,2,72319000,Y,6.0,1.0,NaN,NaN,NaN,Y,OPE,NaN,0,M,30,Technical quality,70,N,1,8447164,2,PROCUREMENT_UNSUCCESSFUL,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"EIOPA/OP/009/2017, Lot 2",Financial Market Input Data for Validation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaT
1,20185,ted.europa.eu/udl?uri=TED:NOTICE:5-2018:TEXT:E...,2018,3,2017-12-22,R209.S2,0,0,N,European Food Safety Authority (EFSA),NaN,Via Carlo Magno 1A,Parma,43126,A1,IT,EU,N,NaN,5,AG,General public\services,N,N,N,S,ITH52,Y,NaN,Y,N,79411000,79411.0,NaN,NaN,N,2.0,1.0,1500000.0,1500000.0,1500000.0,N,OPE,NaN,0,M,NaN,Cheapest price offer/price of tender X---Measu...,40---60,N,2,8447165,NaN,NaN,N,Y,ELT associati di Guaglio Davide e Molinari Pao...,NaN,Via Mantovanella 4---Galleria Crocetta 10/a---...,Mantova---Parma---Parma---Parma---Parma,46100---43126---43126---43123---43122,IT---IT---IT---IT---IT,Y---Y---Y---Y---Y,OC/EFSA/CORSER/2017/03 - FWC 01,Consultancy services to improve efficiency and...,2.0,2.0,0.0,0.0,NaN,1500000.0,1500000.0,1500000.0,N,2017-12-18
2,20185,ted.europa.eu/udl?uri=TED:NOTICE:5-2018:TEXT:E...,2018,3,2017-12-22,R209.S2,0,0,N,European Food Safety Authority (EFSA),NaN,Via Carlo Magno 1A,Parma,43126,A1,IT,EU,N,NaN,5,AG,General public\services,N,N,N,S,ITH52,Y,NaN,Y,N,79411000,79411.0,NaN,NaN,N,2.0,1.0,1500000.0,1500000.0,1500000.0,N,OPE,NaN,0,M,NaN,Cheapest price offer/price of tender X---Measu...,40---60,N,2,8447166,NaN,NaN,N,N,A.I.Erre engineering,NaN,Strada Cavagnari 10,Parma,43126,IT,Y,OC/EFSA/CORSER/2017/03 - FWC02,Consultancy service to improve efficiency and ...,2.0,2.0,0.0,0.0,NaN,1500000.0,1500000.0,1500000.0,N,2017-12-18


In [10]:
df_raw_all[['INFO_ON_NON_AWARD']].value_counts(dropna=False)

INFO_ON_NON_AWARD       
nan                         5212924
PROCUREMENT_UNSUCCESSFUL     794911
PROCUREMENT_DISCONTINUED     190228
Name: count, dtype: int64

In [11]:
df_raw_all[['INFO_ON_NON_AWARD']].value_counts(dropna=False, normalize=True)

INFO_ON_NON_AWARD       
nan                         0.841057
PROCUREMENT_UNSUCCESSFUL    0.128252
PROCUREMENT_DISCONTINUED    0.030692
Name: proportion, dtype: float64

# Downsample balanced data set
Balanced between awarded and non-awarded (and for the non-awarded also balanced between 'PROCUREMENT_UNSUCCESSFUL' and 'PROCUREMENT_DISCONTINUED')

In [22]:
values = {'INFO_ON_NON_AWARD': 'awarded'}
df_raw_all_award = df_raw_all.fillna(value=values)

In [13]:
df_raw_all_award[['INFO_ON_NON_AWARD']].value_counts(dropna=False)

INFO_ON_NON_AWARD       
awarded                     5212924
PROCUREMENT_UNSUCCESSFUL     794911
PROCUREMENT_DISCONTINUED     190228
Name: count, dtype: int64

In [26]:
n_awarded      = 250_000
n_unsuccessful = 125_000
n_discontinued = 125_000

In [28]:
grp_awarded      = df_raw_all_award[df_raw_all_award["INFO_ON_NON_AWARD"] == "awarded"]
grp_unsuccessful = df_raw_all_award[df_raw_all_award["INFO_ON_NON_AWARD"] == "PROCUREMENT_UNSUCCESSFUL"]
grp_discontinued = df_raw_all_award[df_raw_all_award["INFO_ON_NON_AWARD"] == "PROCUREMENT_DISCONTINUED"]

In [29]:
df_raw_all_balanced = (
    pd.concat([
        grp_awarded.sample(n=n_awarded, random_state=42),
        grp_unsuccessful.sample(n=n_unsuccessful, random_state=42),
        grp_discontinued.sample(n=n_discontinued, random_state=42),
    ])
    # .sample(frac=1, random_state=42) # optional shuffle
    .reset_index(drop=True)
)

In [30]:
df_raw_all_balanced[['INFO_ON_NON_AWARD']].value_counts(dropna=False)

INFO_ON_NON_AWARD       
awarded                     250000
PROCUREMENT_UNSUCCESSFUL    125000
PROCUREMENT_DISCONTINUED    125000
Name: count, dtype: int64

In [31]:
df_raw_all_balanced.shape

(500000, 75)

In [32]:
df_raw_all_balanced.to_csv('../raw_data/export_CAN_2023_2018_balanced_500k.tsv', sep='\t', index=False)

In [33]:
# 1_000k sample downsampling
n_awarded      = 500
n_unsuccessful = 250
n_discontinued = 250

df_raw_all_balanced = (
    pd.concat([
        grp_awarded.sample(n=n_awarded, random_state=42),
        grp_unsuccessful.sample(n=n_unsuccessful, random_state=42),
        grp_discontinued.sample(n=n_discontinued, random_state=42),
    ])
    # .sample(frac=1, random_state=42) # optional shuffle
    .reset_index(drop=True)
)

df_raw_all_balanced.to_csv('../raw_data/export_CAN_2023_2018_balanced_1k.tsv', sep='\t', index=False)

In [34]:
df_raw_all_balanced[['INFO_ON_NON_AWARD']].value_counts(dropna=False)

INFO_ON_NON_AWARD       
awarded                     500
PROCUREMENT_UNSUCCESSFUL    250
PROCUREMENT_DISCONTINUED    250
Name: count, dtype: int64

In [35]:
df_raw_all_balanced.shape

(1000, 75)

# Proceed with the 500k file for EDA

In [ ]:
# df_raw_all_balanced